In [ ]:
import control as ct
import numpy as np
print(hasattr(ct, "h2syn"))

: 

In [ ]:
s = ct.tf('s')

# Example plant
G = 1 / (1 + s/(2 * np.pi * 1000))
Gss = ct.tf2ss(G)
Wp_ss = ct.tf2ss(Wp)

# Disturbance weight
Wd = 10 + (2 * np.pi * 200) / s

# Performance weight:
Wp = 1 + (2 * np.pi * 200) / s
Wp_ss = ct.tf2ss(Wp)

# Measurement noise:
N = 0.3/(1 + s / (2 * np.pi * 3000))

# Measurement Dynamics:
[num, den] = ct.pade(600e-6, numdeg=3, n=4)
M = ct.tf(num, den)
M_ss = ct.tf2ss(M)

In [ ]:
def ssdata(sys):
    A, B, C, D = ct.ssdata(sys)
    return np.array(A), np.array(B), np.array(C), np.array(D)

In [ ]:
# Plant
A_G, B_G, C_G, D_G = ssdata(Gss)

# Performance weight
A_Wp, B_Wp, C_Wp, D_Wp = ssdata(Wp_ss)

# Measurement dynamics
A_M, B_M, C_M, D_M = ssdata(M_ss)

# Measurement noise filter
N_ss = ct.tf2ss(N)
A_N, B_N, C_N, D_N = ssdata(N_ss)

assert D_G.shape == (1,1) and D_G[0,0] == 0
assert D_M.shape == (1,1) and D_M[0,0] == 0

nG  = A_G.shape[0]
nWp = A_Wp.shape[0]
nM  = A_M.shape[0]
nN  = A_N.shape[0]

nx = nG + nWp + nM + nN


In [ ]:
A = np.block([
    [A_G,                 np.zeros((nG, nWp)), np.zeros((nG, nM)), np.zeros((nG, nN))],
    [np.zeros((nWp, nG)), A_Wp,                 np.zeros((nWp, nM)), np.zeros((nWp, nN))],
    [B_M @ C_G,           np.zeros((nM, nWp)),  A_M,                np.zeros((nM, nN))],
    [np.zeros((nN, nG)),  np.zeros((nN, nWp)),  np.zeros((nN, nM)), A_N]
])

Bu = np.vstack([
    B_G,
    np.zeros((nWp, 1)),
    np.zeros((nM, 1)),
    np.zeros((nN, 1))
])

Bw = np.block([
    [np.zeros((nG, 2))],
    [np.hstack([B_Wp, np.zeros((nWp,1))])],
    [np.hstack([B_M,  np.zeros((nM,1))])],
    [np.hstack([np.zeros((nN,1)), B_N])]
])

Cz = np.hstack([
    C_G,
    C_Wp,
    np.zeros((1, nM)),
    np.zeros((1, nN))
])

Dz = np.array([[1.0, 0.0, 0.0]])   # [d, n, u]

Cy = np.hstack([
    np.zeros((1, nG)),
    np.zeros((1, nWp)),
    C_M,
    C_N
])

Dy = np.array([[0.0, 0.0, 0.0]])   # [d, n, u]


In [ ]:
# Inputs = [w; u] = [d, n, u]
B = np.hstack([Bw, Bu])

# Outputs = [z; y]
C = np.vstack([Cz, Cy])
D = np.vstack([Dz, Dy])

P = ct.ss(A, B, C, D)

In [ ]:
nmeas = 1   # y
ncon  = 1   # u
K, CL, gamma = ct.h2syn(P, nmeas, ncon)

In [ ]:
print("H2 norm:", gamma)
print("Controller K:")
print(K)